In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import time
import numpy as np

# ==========================================
# 1. CONFIGURATION (The "Tiny" Setup)
# ==========================================
D_MODEL = 24         # Internal dimension
N_HEADS = 4          # 24 / 4 = 6 dim per head
N_LAYERS = 2         # Number of Transformer blocks
FFN_DIM = 48         # Feed-forward expansion
EMB_DIM = 128        # Final JEPA target dimension
MAX_LEN = 32         # Max equation length
BATCH_SIZE = 256
EPOCHS = 30          # Adjust as needed
LR = 1e-3

# Vocabulary Setup
SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>", "<MASK>"]
OPERATORS = ["ADD", "SUB", "MUL", "DIV", "POW", "sin", "cos", "exp", "log"]
VARIABLES = ["x1", "x2", "x3"]
CONSTANTS = [f"<C_{i}>" for i in range(10)]
ALL_TOKENS = SPECIAL_TOKENS + OPERATORS + VARIABLES + CONSTANTS
VOCAB_SIZE = len(ALL_TOKENS)

TOKEN2ID = {t: i for i, t in enumerate(ALL_TOKENS)}
ID2TOKEN = {i: t for i, t in enumerate(ALL_TOKENS)}
PAD_ID, MASK_ID = TOKEN2ID["<PAD>"], TOKEN2ID["<MASK>"]

# ==========================================
# 2. DYNAMIC MASKING DATASET
# ==========================================
class MLMDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Convert string equation to IDs
        tokens = [TOKEN2ID["<BOS>"]] + [TOKEN2ID.get(t, TOKEN2ID["<UNK>"]) for t in self.data[idx].split()] + [TOKEN2ID["<EOS>"]]

        # Padding
        if len(tokens) < MAX_LEN:
            tokens += [PAD_ID] * (MAX_LEN - len(tokens))
        else:
            tokens = tokens[:MAX_LEN]

        tokens = torch.tensor(tokens)
        labels = tokens.clone()

        # Dynamic Masking logic
        # Don't mask Special Tokens (IDs 0-4)
        prob_matrix = torch.full(tokens.shape, 0.15)
        special_tokens_mask = (tokens < 5)
        prob_matrix.masked_fill_(special_tokens_mask, 0.0)

        masked_indices = torch.bernoulli(prob_matrix).bool()

        # Ensure at least one token is masked
        if not masked_indices.any():
            valid_indices = torch.where(~special_tokens_mask)[0]
            masked_indices[valid_indices[torch.randint(0, len(valid_indices), (1,))]] = True

        # Replace with <MASK> ID
        input_ids = tokens.clone()
        input_ids[masked_indices] = MASK_ID

        # Training labels: -100 for non-masked (ignored by CrossEntropy)
        labels[~masked_indices] = -100

        return input_ids, labels, (input_ids != PAD_ID).float()

# ==========================================
# 3. THE 14K PARAMETER MODEL
# ==========================================
class SinusoidalPosEmb(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe_tensor = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe_tensor[:, 0::2] = torch.sin(position * div_term)
        pe_tensor[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe_tensor.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TinyTargetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, D_MODEL, padding_idx=PAD_ID)
        self.pos_enc = SinusoidalPosEmb(D_MODEL, MAX_LEN)

        # Standard Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=FFN_DIM,
            batch_first=True, dropout=0.1, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS)

        # Weight Tying: MLM head uses embedding weights
        self.mlm_bias = nn.Parameter(torch.zeros(VOCAB_SIZE))

        # Projection for JEPA (The 128-dim "Thought" vector)
        self.proj = nn.Linear(D_MODEL, EMB_DIM)

    def forward(self, x, mask):
        x = self.embed(x)
        x = self.pos_enc(x)

        # Pass padding mask to transformer (True means "ignore")
        output = self.transformer(x, src_key_padding_mask=(mask == 0))

        # 1. MLM Logits (Weight Tied)
        logits = F.linear(output, self.embed.weight, self.mlm_bias)

        # 2. Mean Pooling for Sequence Embedding
        # mask is (B, T), unsqueeze to (B, T, 1)
        expanded_mask = mask.unsqueeze(-1)
        pooled = (output * expanded_mask).sum(dim=1) / expanded_mask.sum(dim=1).clamp(min=1e-9)
        equation_emb = self.proj(pooled)

        return logits, equation_emb

# ==========================================
# 4. TRAINING & UTILITIES
# ==========================================

# Dummy data generator for Colab testing
import torch
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = TinyTargetEncoder()

Using device: cuda


/tmp/ipykernel_645/514022606.py:106: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS)


In [10]:

checkpoint = torch.load(
    "/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_jepa_model/tiny_target_encoder_feynman_21pct (1).pt",
    map_location="cuda" # Load to CPU first to avoid device-side assert during initial load
)

# Update global TOKEN2ID, ID2TOKEN, and VOCAB_SIZE from the loaded checkpoint
TOKEN2ID = checkpoint['vocab']
ID2TOKEN = {v: k for k, v in TOKEN2ID.items()}
VOCAB_SIZE = len(TOKEN2ID)

# Now, instantiate the model with the correctly updated VOCAB_SIZE
model = TinyTargetEncoder()
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device) # Move model to GPU after loading state dict
model.eval()

/tmp/ipykernel_645/514022606.py:106: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS)


TinyTargetEncoder(
  (embed): Embedding(27, 24, padding_idx=0)
  (pos_enc): SinusoidalPosEmb()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=24, out_features=24, bias=True)
        )
        (linear1): Linear(in_features=24, out_features=48, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=48, out_features=24, bias=True)
        (norm1): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (proj): Linear(in_features=24, out_features=128, bias=True)
)

In [15]:
ID2TOKEN = {i: t for t, i in TOKEN2ID.items()}

In [ ]:
!mv /content/final_jepa_model /content/drive/MyDrive/symbolic_data

In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

@torch.no_grad()
def build_final_search_library(phase1_model, df_50k, df_feynman):
  phase1_model.eval()
  device = next(phase1_model.parameters()).device

  # 1. Prepare the strings
  # We use the 'expression_prefix_masked' column for 50k
  # We use the 'equation' column for Feynman
  strings_50k = df_50k['expression_prefix_masked'].astype(str).tolist()
  strings_feynman = df_feynman['expression_prefix_masked'].astype(str).tolist()

  all_strings = strings_50k + strings_feynman
  all_embeddings = []

  print(f"Building library for {len(all_strings)} equations...")

  for s in tqdm(all_strings):
      # Tokenize using your SAVED TOKEN2ID
      # Ensure we handle tokens the model might not know with <UNK>
      tokens = ["<BOS>"] + s.split() + ["<EOS>"]
      ids = [TOKEN2ID.get(t, TOKEN2ID["<UNK>"]) for t in tokens]

      # Consistent Padding
      ids = ids[:MAX_LEN] + [TOKEN2ID["<PAD>"]] * max(0, MAX_LEN - len(ids))

      # Create Tensors
      input_ids = torch.tensor(ids).unsqueeze(0).to(device)
      mask = (input_ids != TOKEN2ID["<PAD>"]).float()

      # Generate the 'Target Vector'
      _, emb = phase1_model(input_ids, mask)
      all_embeddings.append(emb.cpu().numpy())

  # 2. Save everything
  combined_matrix = np.concatenate(all_embeddings, axis=0)
  np.save("unified_physics_library.npy", combined_matrix)

  # Save a reference CSV so Index 50001 in the .npy matches Row 50001 in the CSV
  library_df = pd.DataFrame({"equation_string": all_strings})
  library_df.to_csv("/content/drive/MyDrive/symbolic_data/final_data_and_model_lm_jepa_3march/feynman_50k_unified_library_catalog.csv", index=False)

  print(f"Done! Library Shape: {combined_matrix.shape}")
  return combined_matrix

import pandas as pd
df_50k = pd.read_csv("/content/drive/MyDrive/symbolic_data/equations_50k.csv")
df_feynman = pd.read_csv("/content/drive/MyDrive/symbolic_data/feynman_new.csv")
combined_matrix = build_final_search_library(model, df_50k, df_feynman)

Building library for 50100 equations...


100%|██████████| 50100/50100 [00:59<00:00, 842.07it/s]


Done! Library Shape: (50100, 128)


In [ ]:
torch.save(combined_matrix, "/content/drive/MyDrive/symbolic_data/final_data_and_model_lm_jepa_3march/equation_embedding_library_for_retrieval.pt")

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset
import torch

class DecoderWithEmbeddingDataset(Dataset):
    def __init__(self, embeddings, strings):
        self.embeddings = torch.tensor(embeddings).float()

        self.strings = strings

    def __len__(self):
        return len(self.strings)

    def __getitem__(self, idx):
        z = self.embeddings[idx]

        tokens = [TOKEN2ID["<BOS>"]] + \
                 [TOKEN2ID.get(t, TOKEN2ID["<UNK>"]) for t in self.strings[idx].split()] + \
                 [TOKEN2ID["<EOS>"]]

        if len(tokens) < MAX_LEN:
            tokens += [PAD_ID] * (MAX_LEN - len(tokens))
        else:
            tokens = tokens[:MAX_LEN]

        tokens = torch.tensor(tokens)
        input_ids = tokens[:-1].clone()
        labels = tokens[1:].clone()
        labels[labels == PAD_ID] = -100


        return z, input_ids, labels

class JEPADecoder(nn.Module):
  def __init__(self, latent_dim=128, hidden_dim=768, vocab_size=27, max_len=30):
      super().__init__()
      self.max_len = max_len
      self.vocab_size = vocab_size

      # Initial projection: Latent Vector -> RNN Hidden State
      self.latent_to_h0 = nn.Linear(latent_dim, hidden_dim)

      self.embedding = nn.Embedding(vocab_size, hidden_dim)
      self.gru = nn.GRU(hidden_dim, hidden_dim,num_layers=2 ,batch_first=True)
      self.fc_out = nn.Linear(hidden_dim, vocab_size)

  def forward(self, z, target_tokens=None, teacher_forcing_ratio=0.5):
      batch_size = z.size(0)
      h0 = self.latent_to_h0(z)                # (B, hidden_dim)
      h0 = h0.unsqueeze(0)                    # (1, B, hidden_dim)
      h0 = h0.repeat(self.gru.num_layers, 1, 1)  # (num_layers, B, hidden_dim)

      hidden = h0
      # Initialize hidden state from JEPA latent vector
      # hidden = self.latent_to_h0(z).unsqueeze(0) # (1, B, Hidden)

      # Start token
      input_id = torch.full((batch_size, 1), TOKEN2ID["<BOS>"], device=z.device)
      all_logits = []
      seq_len = target_tokens.size(1) if target_tokens is not None else self.max_len

      for t in range(seq_len):


        embedded = self.embedding(input_id) # (B, 1, Hidden)
        output, hidden = self.gru(embedded, hidden)
        logits = self.fc_out(output) # (B, 1, Vocab)
        all_logits.append(logits)

        # Decide next input
        if target_tokens is not None and torch.rand(1).item() < teacher_forcing_ratio:
            input_id = target_tokens[:, t].unsqueeze(1) # Use Ground Truth
        else:
            input_id = logits.argmax(dim=-1) # Use Model Prediction

        if (not self.training) and (input_id == TOKEN2ID["<EOS>"]).all():
            break

      return torch.cat(all_logits, dim=1)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from tqdm import tqdm

def train_decoder(decoder, embeddings, strings, token2id, pad_id, max_len, epochs=50, batch_size=128):
    decoder.to(device)
    optimizer = torch.optim.AdamW(decoder.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    dataset = DecoderWithEmbeddingDataset(embeddings, strings)
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=2, pin_memory=True, shuffle=True)

    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):

        decoder.train()
        total_loss = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")
        for z_batch, input_ids, labels in pbar:
            z_batch = z_batch.to(device, non_blocking=True)
            input_ids = input_ids.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                logits = decoder(z_batch, target_tokens=input_ids)
                loss = criterion(logits.view(-1, decoder.vocab_size), labels.view(-1))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        scheduler.step()
        print(f"Epoch {epoch+1} — Avg Loss: {total_loss/len(loader):.4f}")

    torch.save(decoder.state_dict(), "/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_jepa_model/jepa_decoder_3_march.pt")

In [18]:
combined_matrix = torch.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_data_and_model_lm_jepa_3march/equation_embedding_library_for_retrieval.pt", weights_only=False)
strings = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_data_and_model_lm_jepa_3march/feynman_50k_unified_library_catalog.csv")
equations = strings['equation_string'].to_list()
train_decoder(JEPADecoder(), combined_matrix, equations, TOKEN2ID, PAD_ID, MAX_LEN)

Epoch 1/20: 100%|██████████| 392/392 [00:25<00:00, 15.40it/s, loss=1.9471]


Epoch 1 — Avg Loss: 2.1056


Epoch 2/20: 100%|██████████| 392/392 [00:23<00:00, 16.97it/s, loss=1.9198]


Epoch 2 — Avg Loss: 1.8885


Epoch 3/20: 100%|██████████| 392/392 [00:24<00:00, 15.86it/s, loss=1.8008]


...

Epoch 18 — Avg Loss: 1.5333


Epoch 19/20: 100%|██████████| 392/392 [00:24<00:00, 16.08it/s, loss=1.4060]


Epoch 19 — Avg Loss: 1.5299


Epoch 20/20: 100%|██████████| 392/392 [00:23<00:00, 16.89it/s, loss=1.4497]

Epoch 20 — Avg Loss: 1.5211


In [ ]:
strings = pd.read_csv("/content/drive/MyDrive/symbolic_data/final_data_and_model_lm_jepa_3march/feynman_50k_unified_library_catalog.csv")
strings.iloc[50001]["equation_string"]

'DIV exp DIV POW NEG DIV theta sigma <C_0> <C_0> MUL sqrt MUL <C_0> pi sigma'

In [ ]:
print("Vocab size from TOKEN2ID:", len(TOKEN2ID))
print("Max token ID:", max(TOKEN2ID.values()))

Vocab size from TOKEN2ID: 27
Max token ID: 26


In [20]:
@torch.no_grad()
def test_decode(decoder, embedding, id2token):
    decoder.eval()
    z = torch.tensor(embedding).float().unsqueeze(0).to(device)
    logits = decoder(z, target_tokens=None, teacher_forcing_ratio=0.0)
    ids = logits.argmax(dim=-1).squeeze(0).tolist()
    tokens = [id2token[i] for i in ids]
    # stop at EOS
    if TOKEN2ID["<EOS>"] in ids:
        tokens = tokens[:ids.index(TOKEN2ID["<EOS>"])]
    return " ".join(tokens)

# Test on a few examples
decoder = JEPADecoder(vocab_size=VOCAB_SIZE, max_len=MAX_LEN)
decoder.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_jepa_model/jepa_decoder_3_march.pt"))
decoder.to(device)

ID2TOKEN = {v: k for k, v in TOKEN2ID.items()}
for i in range(5):
    pred = test_decode(decoder, combined_matrix[i], ID2TOKEN)
    print(f"Target: {equations[i]}")
    print(f"Pred:   {pred}\n")

Target: ADD SUB x1 POW x2 x1 MUL SUB <C_0> x2 MUL <C_0> <C_1>
Pred:   ADD SUB POW x2 SUB <C_0> MUL MUL <C_0> x1 MUL x1 <C_1>

Target: POW SUB <C_0> cos x1 e
Pred:   POW cos <C_0> <C_0> <UNK> <UNK>

Target: ADD MUL e cos <C_0> MUL MUL <C_1> x2 POW e x1
Pred:   MUL MUL <UNK> <C_0> <C_0> <UNK> MUL <C_1> <C_1> POW x1 x2

Target: sin MUL MUL MUL x3 x1 MUL <C_0> pi MUL <C_1> <C_1>
Pred:   MUL MUL MUL MUL MUL MUL MUL <UNK> MUL <C_1> <C_1> <C_1>

Target: MUL SUB x1 x1 cos SUB SUB x1 <C_0> arccos x1
Pred:   SUB SUB SUB x1 x1 SUB SUB <C_0> MUL x1 x1



In [13]:
# upgarded 2 gru layers, 512 hidden_dim to 718, used a sheduler, teacher forcing ratio
from tqdm import tqdm
import torch
def train_decoder(decoder, embeddings, strings, token2id, pad_id, max_len, epochs=50, batch_size=128):
    decoder.to(device)
    optimizer = torch.optim.AdamW(decoder.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    dataset = DecoderWithEmbeddingDataset(embeddings, strings)
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=2, pin_memory=True, shuffle=True)

    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        tf_ratio = max(0.3, 1.0 - epoch / epochs)
        decoder.train()
        total_loss = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")
        for z_batch, input_ids, labels in pbar:
            z_batch = z_batch.to(device, non_blocking=True)
            input_ids = input_ids.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                logits = decoder(z_batch, target_tokens=input_ids, teacher_forcing_ratio=tf_ratio)
                loss = criterion(logits.view(-1, decoder.vocab_size), labels.view(-1))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        scheduler.step()
        print(f"Epoch {epoch+1} — Avg Loss: {total_loss/len(loader):.4f}")

    torch.save(decoder.state_dict(), "/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_jepa_model/jepa_decoder_3_march_updraded.pt")

In [14]:
combined_matrix = torch.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_data_and_model_lm_jepa_3march/equation_embedding_library_for_retrieval.pt", weights_only=False)
strings = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_data_and_model_lm_jepa_3march/feynman_50k_unified_library_catalog.csv")
equations = strings['equation_string'].to_list()
train_decoder(JEPADecoder(), combined_matrix, equations, TOKEN2ID, PAD_ID, MAX_LEN)

Epoch 1/50: 100%|██████████| 392/392 [00:37<00:00, 10.37it/s, loss=1.5778]


Epoch 1 — Avg Loss: 1.8084


Epoch 2/50: 100%|██████████| 392/392 [00:35<00:00, 11.15it/s, loss=1.4029]


Epoch 2 — Avg Loss: 1.5629


Epoch 3/50: 100%|██████████| 392/392 [00:36<00:00, 10.62it/s, loss=1.3956]


...

Epoch 48 — Avg Loss: 1.2125


Epoch 49/50: 100%|██████████| 392/392 [00:35<00:00, 11.11it/s, loss=1.2434]


Epoch 49 — Avg Loss: 1.2096


Epoch 50/50: 100%|██████████| 392/392 [00:36<00:00, 10.68it/s, loss=1.1867]


Epoch 50 — Avg Loss: 1.2107


In [15]:
@torch.no_grad()
def test_decode(decoder, embedding, id2token):
    decoder.eval()
    z = torch.tensor(embedding).float().unsqueeze(0).to(device)
    logits = decoder(z, target_tokens=None, teacher_forcing_ratio=0.0)
    ids = logits.argmax(dim=-1).squeeze(0).tolist()
    tokens = [id2token[i] for i in ids]
    # stop at EOS
    if TOKEN2ID["<EOS>"] in ids:
        tokens = tokens[:ids.index(TOKEN2ID["<EOS>"])]
    return " ".join(tokens)

# Test on a few examples
decoder = JEPADecoder(vocab_size=VOCAB_SIZE, max_len=MAX_LEN)
decoder.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/symbolic_data/final_jepa_model/jepa_decoder_3_march_updraded.pt"))
decoder.to(device)

ID2TOKEN = {v: k for k, v in TOKEN2ID.items()}
for i in range(5):
    pred = test_decode(decoder, combined_matrix[i], ID2TOKEN)
    print(f"Target: {equations[i]}")
    print(f"Pred:   {pred}\n")

Target: ADD SUB x1 POW x2 x1 MUL SUB <C_0> x2 MUL <C_0> <C_1>
Pred:   SUB ADD POW x2 x1 x1 MUL MUL <C_0> x2 SUB <C_0> <C_1>

Target: POW SUB <C_0> cos x1 e
Pred:   POW SUB <C_0> cos x1 <UNK>

Target: ADD MUL e cos <C_0> MUL MUL <C_1> x2 POW e x1
Pred:   cos ADD <UNK> <UNK> <C_0> MUL MUL <C_1> x2 POW <UNK> x1

Target: sin MUL MUL MUL x3 x1 MUL <C_0> pi MUL <C_1> <C_1>
Pred:   MUL MUL MUL MUL x1 MUL MUL MUL <UNK> MUL <C_1> <C_1>

Target: MUL SUB x1 x1 cos SUB SUB x1 <C_0> arccos x1
Pred:   SUB SUB x1 x1 x1 SUB SUB x1 <C_0> <UNK> x1



In [ ]:
def compute_token_accuracy(logits, labels):
    """
    logits: (B, T, vocab_size)
    labels: (B, T)
    """

    with torch.no_grad():
        preds = logits.argmax(dim=-1)

        # Ignore -100 positions
        mask = labels != -100

        correct = (preds == labels) & mask
        total = mask.sum()

        accuracy = correct.sum().float() / total.float()
        return accuracy.item()